<a href="https://colab.research.google.com/github/SwRI-IDEA-Lab/butterflai/blob/development%2Fjhamilton/weeks/week_10/10d_train_and_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10 extension — train + evaluate (notebook 10d)

This notebook merges the **training** half (Week 10's Tasks 60–66, formerly
in `10d_extend_and_train.ipynb`) and the **evaluation** half (Tasks 67–70,
formerly in `10e_diffusion_NLL_ablations.ipynb`) into a single Colab-friendly
file. Each Colab session is isolated, so doing both halves in one notebook
means a fresh runtime can pick up where you left off without re-running
setup twice and without risking the `EXPERIMENTS` dict drifting between the
two halves.

## How to use this notebook

There is **one flag at the top** that determines what the notebook does:

- `MODE = "train"` — runs the data-augmentation cells (Tasks 60–63), the
  experiment-menu setup (Tasks 64–65), the training loop (Task 66), and the
  visual sanity check. The training loop is **idempotent**: it skips any
  experiment whose `ckpt_<name>.ckpt` already exists. Run the notebook many
  times with one new entry enabled in `ENABLED_EXPERIMENTS` per session.
- `MODE = "eval"` — skips training, jumps into the NLL ablation pipeline
  (Tasks 67–69), and scores every `ckpt_E*.ckpt` it finds in this directory.

**Recipe:** run with `MODE = "train"` repeatedly (one new experiment per
session) until you have all the checkpoints you want, then flip to
`MODE = "eval"` and run the notebook once to score them.

[**↓ Jump to evaluation (Tasks 67–70)**](#eval-mode)

> The **test split is reserved for the PI**. Every data-loading cell in this
> notebook filters to `split in {"train", "val"}`. Do not change that.


In [1]:
MODE = "eval"   # set to "eval" once you have ckpt_E*.ckpt files to score

assert MODE in ("train", "eval"), f"MODE must be 'train' or 'eval', got {MODE!r}"
TRAIN_MODE = (MODE == "train")
EVAL_MODE  = (MODE == "eval")
print(f"MODE = {MODE!r}  (TRAIN_MODE={TRAIN_MODE}, EVAL_MODE={EVAL_MODE})")


MODE = 'eval'  (TRAIN_MODE=False, EVAL_MODE=True)


In [2]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
repo_path = "/content/butterflai"

# Use this path if working locally
# repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


  Installing from /content/butterflai/requirements.txt...

🦋 ButterflAI environment ready
   Runtime  : Google Colab
   Device   : cpu
   Seed     : 42


{'in_colab': True,
 'device': device(type='cpu'),
 'seed': 42,
 'drive_mounted': False,
 'data_path': None}

In [3]:
!pip install pytorch_lightning

In [4]:
import os, sys, glob, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import copy # Added to resolve NameError: name 'copy' is not defined

from scipy.stats import norm as sp_norm
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import re
import os
import importlib
import subprocess
import torch
import numpy as np

def patch_block_cond_concat_function(content):
    # Define the block_cond_concat function to be inserted
    # This function extracts raw conditions from blocks, normalizes them, and concatenates
    block_cond_concat_code = '''
def block_cond_concat(hcs, lit_model, original_cfg):
    """Builds concatenated conditional tensors for each block in hcs,
    using normalization parameters and Fourier settings from the lit_model's hparams.

    Args:
        hcs: List of hemicycle structures, each containing 'blocks'.
        lit_model: The trained LightningModule, containing hparams for normalization and cond config.
        original_cfg: The experiment configuration dictionary (used as fallback for consumed_keys).

    Returns:
        List of torch.Tensor, where each tensor corresponds to a block in hcs
        and contains the concatenated and normalized conditional vectors
        for all items within that block.
        Shape of each tensor: (num_items_in_block, cond_dim).
    """
    all_block_conds = []

    # Use hparams from the loaded model to ensure consistency with training
    model_hparams = lit_model.hparams
    # Get consumed_keys and fourier setting from model's hparams, fallback to original_cfg if not present
    consumed_keys = model_hparams.get("consumed_keys", original_cfg.get("consumed_keys"))
    fourier_enabled = model_hparams.get("fourier", False) # Default to False if not in hparams
    # Crucial change: Use model_hparams for fourier_supported_scalars for consistency
    fourier_supported_scalars_map = model_hparams.get("fourier_supported_scalars", {})

    if consumed_keys is None:
        raise ValueError("Consumed keys not found in model hparams or original config.")

    # Access GROUP_COLS from the LightningModule class (should be consistent)
    group_cols_map = lit_model.GROUP_COLS # Assumes GROUP_COLS is a class attribute

    for hemicycle in hcs:
        for block in hemicycle['blocks']:
            cond_components = []

            for group_key_prefix in consumed_keys:
                group_key = group_key_prefix.replace("cond_", "") # Convert 'cond_base' to 'base'
                if group_key not in group_cols_map:
                    # This should ideally not happen if consumed_keys is consistent
                    raise ValueError(f"Unknown conditional group '{group_key}' specified in model hparams' consumed_keys.")

                # Extract raw values from groups_raw dictionary in the block
                raw_cond_values_np = block['groups_raw'][group_key]
                raw_cond_values = torch.tensor(raw_cond_values_np, dtype=torch.float32)

                # Prepare for normalization or direct use
                # Default to raw values, will be replaced if normalization or fourier applies
                current_component_values = raw_cond_values.flatten()

                mean_key = f"cond_{group_key}_means"
                std_key = f"cond_{group_key}_stds"

                # Check if normalization parameters exist in model hparams/buffers
                if mean_key in model_hparams and std_key in model_hparams:
                    cond_mean = model_hparams[mean_key].to(raw_cond_values.device)
                    cond_std = model_hparams[std_key].to(raw_cond_values.device)
                    current_component_values = (raw_cond_values - cond_mean.squeeze()) / (cond_std.squeeze() + 1e-6)
                else:
                    print(f"Warning: Normalization parameters for '{group_key}' not found in model hparams. Using raw values. (This may affect NLL calculation but not dimension.)")
                    # If normalization params are missing, use raw values (already assigned as default)


                # Apply Fourier features if enabled for this model and supported for this group's scalars
                if fourier_enabled and group_key in fourier_supported_scalars_map:
                    fourier_scalars_in_group = fourier_supported_scalars_map[group_key]
                    original_cols_in_group = group_cols_map[group_key]
                    processed_fourier_components = []

                    for i, col_name in enumerate(original_cols_in_group):
                        val = current_component_values[i] # Expecting current_component_values to be 1D for a single item
                        if col_name in fourier_scalars_in_group:
                            # Apply sin/cos transformation
                            processed_fourier_components.append(torch.sin(val * torch.pi))
                            processed_fourier_components.append(torch.cos(val * torch.pi))
                        else:
                            # Keep as is
                            processed_fourier_components.append(val)

                    if processed_fourier_components:
                        cond_components.append(torch.stack(processed_fourier_components).flatten()) # Flatten before appending
                    else:
                        # If Fourier was enabled but no scalars were processed (e.g., empty group, or no scalars for fourier)
                        cond_components.append(current_component_values.flatten()) # Fallback to original component values
                else:
                    # No Fourier or not supported, just append the current (normalized/raw) values
                    cond_components.append(current_component_values.flatten())

            if cond_components:
                # Concatenate all components for this block.
                # Ensure each component is 1D before concatenating
                all_block_conds.append(torch.cat(cond_components, dim=-1).unsqueeze(0))
            else:
                all_block_conds.append(torch.empty(1, 0, dtype=torch.float32))

    return all_block_conds
'''

    # Always append to the end of the file to avoid indentation issues.
    # The module reloading logic in cell 64eb3a17 is designed to handle this.
    print("Appending `block_cond_concat` function definition to end of file.")
    return content + '\n\n' + block_cond_concat_code + '\n'

# --- Paths and patching logic ---
repo_path = "/content/butterflai"
conditioned_infrastructure_path = os.path.join(repo_path, "weeks", "week_10", "conditioned_infrastructure.py")

print(f"Resetting {conditioned_infrastructure_path} to git HEAD state before patching `block_cond_concat`.")
relative_path = os.path.relpath(conditioned_infrastructure_path, repo_path)
subprocess.run(["git", "checkout", relative_path], cwd=repo_path, check=True)

with open(conditioned_infrastructure_path, 'r') as f:
    original_content = f.read()

patched_content = patch_block_cond_concat_function(original_content)

with open(conditioned_infrastructure_path, 'w') as f:
    f.write(patched_content)

print(f"File {conditioned_infrastructure_path} patched successfully with `block_cond_concat`.")

# --- No module reload/re-import here; rely on cell 64eb3a17 to handle that ---

Resetting /content/butterflai/weeks/week_10/conditioned_infrastructure.py to git HEAD state before patching `block_cond_concat`.
Appending `block_cond_concat` function definition to end of file.
File /content/butterflai/weeks/week_10/conditioned_infrastructure.py patched successfully with `block_cond_concat`.


In [30]:
import os, sys
import importlib
import glob # Explicitly import glob here to ensure availability

# Define the absolute paths for week_08, week_09 and week_10
week08_path = os.path.abspath(os.path.join(repo_path, "weeks", "week_08"))
week09_path = os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))
week10_path = os.path.abspath(os.path.join(repo_path, "weeks", "week_10"))

# Clean up sys.path from previous runs to ensure consistent order
# Create a new list of paths to rebuild sys.path, ensuring repo_path is first.
new_sys_path = [repo_path]

# Add week paths after repo_path. The order here helps resolve potential conflicts
# for modules that might exist in multiple week directories (e.g., conditioned_infrastructure.py)
# and for direct imports like 'unconditioned_infrastructure'.
if week10_path not in new_sys_path:
    new_sys_path.append(week10_path)
if week09_path not in new_sys_path:
    new_sys_path.append(week09_path)
if week08_path not in new_sys_path:
    new_sys_path.append(week08_path)

# Add any other existing sys.path entries that are not one of our specific paths
for p in sys.path:
    if p not in new_sys_path:
        new_sys_path.append(p)

sys.path = new_sys_path

# Clear relevant modules from sys.modules to force a fresh import from the updated sys.path.
modules_to_clear = [
    'conditioned_infrastructure',
    'unconditioned_infrastructure',
    'butterflAI_model',
    'weeks.week_10.conditioned_infrastructure',
    'weeks.week_09.conditioned_infrastructure',
    'weeks.week_09.unconditioned_infrastructure',
    'weeks.week_08.butterflAI_model',
]
for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

# Import the main conditioned_infrastructure module first, then reload it
# to ensure all internal definitions are fresh after patching and sys.modules clearing.
import weeks.week_10.conditioned_infrastructure as conditioned_infrastructure_module
importlib.reload(conditioned_infrastructure_module)

# Explicitly change the current working directory to the repo_path.
# This is crucial for modules that infer paths based on os.getcwd().
original_cwd = os.getcwd()
os.chdir(repo_path)

# Now, perform the imports using the reloaded module.
# Use find_week10_artifacts directly from the reloaded module
paths = conditioned_infrastructure_module.find_week10_artifacts(extra_required=[
    "data/composite_sunspot_groups_peak_area.csv",
])
print(f"using conditioned_infrastructure from: {paths['conditioned_py']}")

# Restore the original working directory after the call.
os.chdir(original_cwd)

# These imports should now work correctly as their parent directories are in sys.path.
from weeks.week_09.unconditioned_infrastructure import make_cosine_schedule
from weeks.week_10.conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    ExtendedConditionalDiffusionLightning,
    train_experiment,
    load_trained_experiment,
    sample_conditional_extended,
    build_model,
    block_cond_concat,
    k_run_combined,
    discover_experiment_checkpoints,
)
from weeks.week_08.butterflAI_model import ButterflAIModel

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the v1 parquet — we extend it but never modify it.
# Test split is reserved for the PI.
windows_v1 = pd.read_parquet(paths["parquet_v1"])
windows_v1 = windows_v1.loc[windows_v1["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v1 parquet (train+val only): {len(windows_v1)} rows")
print(f"  splits: {windows_v1['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v1['cycle'].unique())}")
print(f"v2 parquet target: {PARQUET_V2}")
print(f"device           : {device}")

using conditioned_infrastructure from: /content/butterflai/weeks/week_10/conditioned_infrastructure.py
v1 parquet (train+val only): 287 rows
  splits: {'train': 232, 'val': 55}
  cycles: [np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
v2 parquet target: /content/butterflai/weeks/week_10/diffusion_windows_v2.parquet
device           : cpu


---
## Part A — Build the augmented parquet

We're going to add three families of new conditioning columns to the
v1 parquet:

1. **Cycle and hemisphere identifiers** — a normalized cycle number and
   a north/south indicator.
2. **Opposite-hemisphere summaries** — for each window, the
   *contemporaneous* opposite-hemisphere activity. This is not leakage:
   contemporaneous opposite-hemisphere activity is operationally
   observable (an operational forecaster on the day of the same window
   would have it).
3. **Smoothed-area trajectory** — the past `K` windows of `area_smoothed`
   for the same hemicycle, as a short autoregressive history.

The build cells **rewrite `diffusion_windows_v2.parquet` every run**. We
deliberately do *not* gate on file existence — if you change how a
column is computed and don't see the change downstream, the most
common explanation is "the file was cached." We avoid that failure
mode by always rewriting.


---
## Task 60 — Cycle and hemisphere identifiers

Add two new columns to the dataframe:

- `cycle_norm`: cycle number rescaled to roughly `[-1, +1]` over the
  train range. The point of normalization is not to be exactly in
  `[-1, +1]` — it's to put the input on the same numerical scale as the
  other conditioning vectors so the network doesn't have to learn an
  outsized weight for it.
- `hemi_id`: `+1` for north, `-1` for south.

Both are cheap and let downstream experiments test whether
*structural* per-cycle / per-hemisphere effects survive once amplitude
is controlled for.


In [8]:
# Task 60 — add cycle_norm and hemi_id.

windows_aug = windows_v1.copy()

# TODO: compute cycle_norm. Use train-set cycle range so this is well
# defined for both splits. Aim for the train cycle range to map roughly
# onto [-1, +1].
_train_cycles = windows_aug.loc[windows_aug["split"] == "train", "cycle"]
_cmin, _cmax  = _train_cycles.min(), _train_cycles.max()
windows_aug["cycle_norm"] = 2.0 * (windows_aug["cycle"] - _cmin) / (_cmax - _cmin) - 1.0

# TODO: compute hemi_id. +1 north, -1 south.
windows_aug["hemi_id"] = np.where(windows_aug["hemisphere"] == "north", 1.0, -1.0)

print("cycle_norm range:", windows_aug["cycle_norm"].min(), windows_aug["cycle_norm"].max())
print("hemi_id values  :", windows_aug["hemi_id"].unique())


cycle_norm range: -1.0 1.0
hemi_id values  : [-1.  1.]


---
## Task 61 — Opposite-hemisphere conditioning

For each window in hemisphere *h* at time `tau_center`, attach the
contemporaneous opposite-hemisphere activity summary:

- `opp_area_smoothed` — opposite hemisphere's `area_smoothed`
- `opp_mu_universal`  — opposite hemisphere's `mu_universal`
- `opp_amplitude`     — opposite hemisphere's `amplitude`
- `opp_valid`         — 1 if a matching opposite row was found at the
  same `(cycle, tau_center)`, else 0.

When `opp_valid == 0` (no matching opposite row), impute with the
**train-set mean** of each opposite-* column. This way the network always
sees a defined input; downstream you can decide whether to gate on the
mask.

**Implementation hint:** the cleanest way is a self-merge of the
dataframe with itself: produce a "right side" with `hemisphere`
flipped and renamed columns, then merge on `(cycle, tau_center)`.


In [9]:
# Task 61 — attach contemporaneous opposite-hemisphere summaries.

# Identify columns that this task will create and ensure they don't exist
# in windows_aug from a previous run to prevent merge errors.
cols_to_drop = ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude", "opp_valid", "_opp_match", "_tau_center_rounded"]
for col in cols_to_drop:
    if col in windows_aug.columns:
        windows_aug = windows_aug.drop(columns=[col])

# Ensure the 'tau_center' column is consistently named for this operation.
# If a previous merge renamed it to 'tau_center_x', rename it back to 'tau_center'.
if 'tau_center_x' in windows_aug.columns and 'tau_center' not in windows_aug.columns:
    windows_aug = windows_aug.rename(columns={'tau_center_x': 'tau_center'})
# If a 'tau_center_y' column also exists (less likely to be the primary one needed), drop it.
if 'tau_center_y' in windows_aug.columns:
    windows_aug = windows_aug.drop(columns=['tau_center_y'])

_flip = {"north": "south", "south": "north"}

# Round tau_center to address potential floating point discrepancies in merging
# We'll use a temporary rounded column for the merge.
precision = 1 # Reduced precision as requested by the user
windows_aug['_tau_center_rounded'] = windows_aug['tau_center'].round(precision)

# Create the _right DataFrame. This DataFrame will hold the data from the opposite hemisphere
# and will be merged into `windows_aug`. It needs:
# 1. Merge keys: 'cycle', 'hemisphere' (flipped), '_tau_center_rounded'
# 2. Data columns: 'area_smoothed', 'mu_universal', 'amplitude' (which will be renamed to opp_something)
_right_source = windows_aug.loc[:, ["cycle", "tau_center", "hemisphere", # Use the canonical tau_center for source
                         "area_smoothed", "mu_universal", "amplitude"]]

# Create a temporary rounded tau_center for _right_source, so its 'tau_center' isn't conflicting.
_right_source['_tau_center_rounded'] = _right_source['tau_center'].round(precision)

# Now rename the data columns to their 'opp_' prefixed names.
_right = _right_source.rename(columns={
    "area_smoothed": "opp_area_smoothed",
    "mu_universal":  "opp_mu_universal",
    "amplitude":     "opp_amplitude",
})

# Now, flip the 'hemisphere' column in _right.
# This means if _right originally had a row for 'north' data, its 'hemisphere'
# is now 'south'. If it was 'south', it's now 'north'.
# This ensures that when we merge on (cycle, _tau_center_rounded, hemisphere),
# a 'north' row in windows_aug will find a 'north' row in _right, but that
# _right row's data actually originated from the 'south' hemisphere.
_right["hemisphere"] = _right["hemisphere"].map(_flip)

# Drop the original `tau_center` column from _right to avoid suffixing during the merge,
# as `_tau_center_rounded` will be used as the merge key.
_right = _right.drop(columns=['tau_center'])

# Merge windows_aug with the _right DataFrame.
# The merge is on (cycle, _tau_center_rounded, hemisphere).
# The 'hemisphere' in _right now acts as the 'target' hemisphere for matching.
windows_aug = windows_aug.merge(
    _right, on=["cycle", "_tau_center_rounded", "hemisphere"],
    how="left", indicator="_opp_match",
)
windows_aug["opp_valid"] = (windows_aug["_opp_match"] == "both").astype(np.float32)

# Drop temporary merge columns. The original 'tau_center' column in windows_aug should be preserved.
windows_aug = windows_aug.drop(columns=["_opp_match", "_tau_center_rounded"])

# Impute missing opp_* values with train-set means.
_opp_cols = ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"]
_train_mask = (windows_aug["split"] == "train") & (windows_aug["opp_valid"] == 1.0)
for _c in _opp_cols:
    _mean = windows_aug.loc[_train_mask, _c].mean()
    windows_aug[_c] = windows_aug[_c].fillna(_mean)

print(f"opp_valid coverage: {windows_aug['opp_valid'].mean():.3f}")
print(f"opp_area_smoothed (train, valid): "
      f"mean={windows_aug.loc[_train_mask, 'opp_area_smoothed'].mean():.3f}, "
      f"std={windows_aug.loc[_train_mask, 'opp_area_smoothed'].std():.3f}")

opp_valid coverage: 0.195
opp_area_smoothed (train, valid): mean=86.506, std=36.097


---
## Task 62 — Smoothed-area trajectory (cycle history)

For each window, attach the previous `K` values of `area_smoothed`
from the same hemicycle (same `cycle` AND same `hemisphere`), ordered
chronologically by `tau_center`. Columns: `area_lag1`, `area_lag2`, …,
`area_lagK`.

`area_lag1` is the *previous* window's smoothed area; `area_lagK` is
the one K steps back. Boundary windows (near the start of a hemicycle,
where fewer than K prior windows exist) get train-set-mean imputation
and a `traj_valid` column set to 0 for that row.

The point: cycle *history* is information that amplitude alone misses.
A window 6 months into a strong cycle and a window 6 months from the
end of a strong cycle have similar amplitude but very different
trajectory.

**Implementation hint:** sort within each hemicycle by `tau_center`,
then shift the `area_smoothed` series by 1, 2, …, K.


In [10]:
# Task 62 — attach the smoothed-area trajectory (K lagged values).

K_LAGS = 4

# TODO: within each (cycle, hemisphere) group, sort by tau_center and
#       produce K columns of shifted area_smoothed.
_sorted = windows_aug.sort_values(["cycle", "hemisphere", "tau_center"])

_lag_cols = [f"area_lag{k}" for k in range(1, K_LAGS + 1)]
for k, col in enumerate(_lag_cols, start=1):
    _sorted[col] = _sorted.groupby(["cycle", "hemisphere"])["area_smoothed"].shift(k)

# A row is traj_valid iff *all* K lags exist.
_sorted["traj_valid"] = (~_sorted[_lag_cols].isna().any(axis=1)).astype(np.float32)

# TODO: impute boundary NaNs with train-set means.
_train_mask = (_sorted["split"] == "train") & (_sorted["traj_valid"] == 1.0)
for col in _lag_cols:
    _mean = _sorted.loc[_train_mask, col].mean()
    _sorted[col] = _sorted[col].fillna(_mean)

windows_aug = _sorted.sort_index()  # restore original row order

print(f"traj_valid coverage: {windows_aug['traj_valid'].mean():.3f}")
print(f"lag columns: {_lag_cols}")


traj_valid coverage: 0.721
lag columns: ['area_lag1', 'area_lag2', 'area_lag3', 'area_lag4']


---
## Task 63 — Write `diffusion_windows_v2.parquet` and sanity-check

Write the augmented dataframe. Sanity checks before we trust it
downstream:

- Row count unchanged from v1 (we did not gain or lose any windows).
- Every original v1 column is preserved bit-for-bit.
- New cond columns are finite **wherever the validity mask says they
  should be**.
- `split` column is unchanged.


In [11]:
# Task 63 — write v2 + sanity checks.

# Sanity 1: row count.
assert len(windows_aug) == len(windows_v1), \
    f"row count drifted: {len(windows_aug)} vs v1 {len(windows_v1)}"

# Sanity 2: every v1 column preserved bit-for-bit.
for c in windows_v1.columns:
    assert c in windows_aug.columns, f"missing v1 column {c}"
    if windows_v1[c].dtype.kind in "fc":
        assert np.allclose(windows_v1[c].to_numpy(),
                           windows_aug[c].to_numpy(), equal_nan=True), c
    else:
        assert (windows_v1[c].astype(str).to_numpy()
                == windows_aug[c].astype(str).to_numpy()).all(), c

# Sanity 3: new columns finite where the validity masks allow.
NEW_COND_COLS = ["cycle_norm", "hemi_id",
                 "opp_area_smoothed", "opp_mu_universal", "opp_amplitude",
                 *[f"area_lag{k}" for k in range(1, K_LAGS + 1)]]
for c in NEW_COND_COLS:
    assert windows_aug[c].notna().all(), f"NaN remains in {c}"

# Sanity 4: split column unchanged.
assert (windows_aug["split"].to_numpy() == windows_v1["split"].to_numpy()).all()

# Always rewrite. Never cache.
windows_aug.to_parquet(PARQUET_V2, index=False)
print(f"wrote {PARQUET_V2}  ({len(windows_aug)} rows, {len(windows_aug.columns)} cols)")
print(f"new cond columns: {NEW_COND_COLS}")


wrote /content/butterflai/weeks/week_10/diffusion_windows_v2.parquet  (287 rows, 49 cols)
new cond columns: ['cycle_norm', 'hemi_id', 'opp_area_smoothed', 'opp_mu_universal', 'opp_amplitude', 'area_lag1', 'area_lag2', 'area_lag3', 'area_lag4']


---
## Part B — wandb setup and the experiment menu

### Task 64 — Per-student wandb project

Each student gets their **own** wandb project. Replace the placeholder
strings below with your handle. Every training run in this notebook
logs to that project with the experiment ID as the run name; you can
compare all your variants on a single dashboard.

If wandb is unavailable in your environment, training will fall back to
a local CSV logger automatically. The assertion guard runs in both
modes so eval mode also tells you if you forgot to personalize the
project name.


In [15]:
# Task 64 — wandb identity. EDIT THESE.

WANDB_PROJECT = "butterflai-w10ext-jhamilton"
WANDB_ENTITY  = "butterflai-project"      # set to None if you don't use teams

assert "jhamilton58" not in WANDB_PROJECT, \
    "Set WANDB_PROJECT to your own project name before training."


In [ ]:
import wandb

# Log in to Weights & Biases. You will be prompted to enter your API key.
wandb.login()

### Task 65 — Design your own experiments

The Week 10 baseline (E0) reproduces the existing conditional
diffusion on the v2 parquet — no new knobs. Everything beyond it is
your call. Each variant you propose should change **one knob** from
the previous run and answer **one question**.

The knobs available are:

- **Cond groups** (`groups` / `consumed_keys`): `base`, plus any of
  `cyclehemi`, `opp`, `traj`.
- **Architecture** (`arch`): `concat` or `film`.
- **Classifier-free guidance**: `cond_dropout_p=0.1` at training time;
  10e sweeps the guidance weight at sampling.
- **Fourier lifting**: `fourier=True` lifts cond scalars via sin/cos.

The menu below escalates roughly by effort-per-insight. Pick what's
interesting, add a new entry to `EXPERIMENTS`, and progress one
variant per session.

**Level 1 — same cond, change the channel.**
Add one of the new cond groups (`cyclehemi`, `opp`, `traj`) to E0's
`consumed_keys`. *Does the diffusion's val NLL drop when given more
information, with the architecture held fixed?*

**Level 2 — same information, change the mechanism.**
Switch `arch` from `concat` to `film` while keeping `cond_base` only.
*Does the modulation mechanism alone close the gap with classical?*

**Level 3 — best information × best mechanism.**
Combine your best Level 1 cond set with FiLM. *Is the combined gain
additive, or did Level 2 already capture it?*

**Level 4 — guidance.**
Set `cond_dropout_p=0.1` and train. Sampling guidance is swept in 10e.
*Can sharpening the conditional density buy you margin over Level 3?*

**Level 5 — Fourier lifting.**
Set `fourier=True`. *Does sin/cos lifting of the cond scalars help the
network represent boundaries?*

You can go further — bump `hidden_dim` / `n_layers`, raise `K_LAGS`
back in Task 62, pair lagged opposite-hemisphere with trajectory, or
anything else you can defend. Different students should diverge here;
results pool in 10e.

In [47]:
# Task 65 — experiment specs. Start with the baseline; add new entries
# below as you escalate (see the markdown above). See
# conditioned_infrastructure.build_model for the recognized keys.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "fourier_supported_scalars": {}, # Explicitly add this for clarity
    "cond_dropout_p": 0.0,
    "max_epochs":     20000,
    "lr":             1e-3,
    "batch_size":     64,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    # Original E0 had fourier=False, but the error indicates it was trained with Fourier on tau_center.
    # Updating E0 config to match the likely training configuration of the provided checkpoint.
    "E0": _spec(fourier=True, fourier_supported_scalars={"base": ["tau_center"]}), # baseline — now correctly reflects Fourier features for tau_center
    # Add your own variants below, e.g.:
    "E1": _spec(consumed_keys=["cond_base", "cond_cyclehemi"], # Level 1: Add cyclehemi
                groups=["base", "cyclehemi"]),
    "E2": _spec(consumed_keys=["cond_base", "cond_traj"], # Level 1: Add traj
                groups=["base", "traj"]),
    "E3": _spec(arch="film"), # Level 2: Switch to FiLM architecture
    "E4": _spec(arch="film", # Level 3: Combine best Level 1 (assuming cyclehemi) with FiLM
                consumed_keys=["cond_base", "cond_cyclehemi"],
                groups=["base", "cyclehemi"])
}

for name, cfg in EXPERIMENTS.items():
    print(f"{name}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")

E0: arch=concat  consumed=['cond_base']  fourier=True  cond_dropout_p=0.0
E1: arch=concat  consumed=['cond_base', 'cond_cyclehemi']  fourier=False  cond_dropout_p=0.0
E2: arch=concat  consumed=['cond_base', 'cond_traj']  fourier=False  cond_dropout_p=0.0
E3: arch=film    consumed=['cond_base']  fourier=False  cond_dropout_p=0.0
E4: arch=film    consumed=['cond_base', 'cond_cyclehemi']  fourier=False  cond_dropout_p=0.0


---
## Part C — Disciplined sweep (TRAIN MODE)

### Task 66 — Enable a subset and train

Discipline:

- Add at most **one new experiment per session** beyond the baseline.
  Two-knob-at-a-time changes make the 10e diff impossible to read.
- Each enabled experiment logs to your wandb project under its name
  (`E0`, `E1`, …); compare them on a single dashboard.
- The loop skips checkpoints that already exist on disk, so re-running
  the notebook does not retrain unless you delete the file.

*The cells below only execute when `MODE == "train"`.*


In [13]:
if not TRAIN_MODE:
    print("Skipping Task 66 training loop (MODE=eval). Switch MODE to \"train\" to run this cell.")
else:
    # Task 66 — enable, then train. EDIT THIS LIST.

    ENABLED_EXPERIMENTS = ["E0"]   # add one experiment per session

    for _name in ENABLED_EXPERIMENTS:
        if _name not in EXPERIMENTS:
            raise KeyError(f"unknown experiment {_name!r}; defined: {list(EXPERIMENTS)}")
        train_experiment(
            name=_name, cfg=EXPERIMENTS[_name],
            windows_aug=windows_aug, ckpt_dir=CKPT_DIR,
            wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
            alpha_np=alpha_np, sigma_np=sigma_np, T=T,
            bin_centers=BIN_CENTERS, bin_width=BIN_WIDTH,
        )

Skipping Task 66 training loop (MODE=eval). Switch MODE to "train" to run this cell.


In [14]:
import os
import shutil
import glob

if not TRAIN_MODE:
    print("Skipping checkpoint saving (MODE=eval). Switch MODE to \"train\" and enable experiments to save checkpoints.")
else:
    # Define the destination directory in My Drive
    drive_checkpoint_dir = os.path.join('/content/drive/MyDrive', 'butterflai_checkpoints')

    # Create the directory if it doesn't exist
    os.makedirs(drive_checkpoint_dir, exist_ok=True)
    print(f"Ensured directory exists: {drive_checkpoint_dir}")

    # Get a list of all checkpoint files in CKPT_DIR
    all_ckpts_in_dir = glob.glob(os.path.join(CKPT_DIR, 'ckpt_E*.ckpt'))

    # Filter for checkpoints that belong to ENABLED_EXPERIMENTS
    checkpoints_to_save = []
    for experiment_name in ENABLED_EXPERIMENTS:
        for ckpt_path in all_ckpts_in_dir:
            if f'ckpt_{experiment_name}.ckpt' in os.path.basename(ckpt_path):
                checkpoints_to_save.append(ckpt_path)

    if checkpoints_to_save:
        print(f"Saving {len(checkpoints_to_save)} checkpoints to {drive_checkpoint_dir}:")
        for ckpt_path in checkpoints_to_save:
            dest_path = os.path.join(drive_checkpoint_dir, os.path.basename(ckpt_path))
            shutil.copy(ckpt_path, dest_path)
            print(f"  Copied {os.path.basename(ckpt_path)}")
    else:
        print("No checkpoints found for the enabled experiments to save.")

Skipping checkpoint saving (MODE=eval). Switch MODE to "train" and enable experiments to save checkpoints.


---
## Part D — Visual sanity check on the most recent training

Sample a small batch of validation conditioning vectors and overlay
the diffusion's generated residuals against the ground truth. This is
a "did training collapse?" check — not a quantitative comparison. The
real evaluation lives in the EVAL MODE section below.


In [15]:
if not TRAIN_MODE:
    print("Skipping Part D visual sanity check (MODE=eval). Switch MODE to \"train\" to run this cell.")
else:
    # Part D — quick overlay for the most recently trained checkpoint.

    if not ENABLED_EXPERIMENTS:
        print("No experiments were trained this session — nothing to visualize.")
    else:
        _name = ENABLED_EXPERIMENTS[-1]
        _cfg  = EXPERIMENTS[_name]
        lit, _, val_ds, _ = load_trained_experiment(
            _name, _cfg, windows_aug, CKPT_DIR, alpha_np, sigma_np,
        )

        n_show = 4
        cond_concat = torch.cat(
            [torch.stack([val_ds[i][k] for i in range(n_show)])
             for k in _cfg["consumed_keys"]],
            dim=-1,
        )
        truth = torch.stack([val_ds[i]["r_clean"] for i in range(n_show)]).numpy()
        truth_phys = truth * val_ds.bin_stds.numpy() + val_ds.bin_means.numpy()
        samples = sample_conditional_extended(lit, cond_concat, guidance_w=0.0).cpu().numpy()

        fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3.2), sharey=True)
        for i, ax in enumerate(axes):
            w = BIN_WIDTH * 0.4
            ax.bar(BIN_CENTERS - w / 2, truth_phys[i], width=w, color="C0", label="truth")
            ax.bar(BIN_CENTERS + w / 2, samples[i],    width=w, color="C2", label="sampled")
            ax.axhline(0, color="k", lw=0.4)
            ax.set_title(f"val window {i}")
            ax.set_xlabel("|latitude| (°)")
        axes[0].set_ylabel("residual"); axes[0].legend()
        fig.suptitle(f"{_name}: visual sanity check")
        fig.tight_layout(); plt.show()

Skipping Part D visual sanity check (MODE=eval). Switch MODE to "train" to run this cell.


In [16]:
# Define the base directory for checkpoints in Google Drive
drive_checkpoint_dir = '/content/drive/MyDrive/butterflai_checkpoints'

# Define the list of experiment names to load
# These are the checkpoints that were previously saved to Drive
EXPERIMENT_NAMES_TO_LOAD = ['E0', 'E1', 'E2', 'E3', 'E4']

# Dictionary to store the loaded models
loaded_checkpoints = {}

print(f"Loading {len(EXPERIMENT_NAMES_TO_LOAD)} checkpoints from {drive_checkpoint_dir}...")

for name in EXPERIMENT_NAMES_TO_LOAD:
    if name not in EXPERIMENTS:
        print(f"Warning: Experiment '{name}' not found in EXPERIMENTS dictionary. Skipping.")
        continue

    cfg = EXPERIMENTS[name]
    try:
        # load_trained_experiment requires ckpt_dir to be the location of the checkpoint files
        lit, _, val_ds, train_ds = load_trained_experiment(
            name, cfg, windows_aug, drive_checkpoint_dir, alpha_np, sigma_np
        )
        loaded_checkpoints[name] = lit
        print(f"Successfully loaded checkpoint for experiment '{name}'.")
    except Exception as e:
        print(f"Error loading checkpoint for experiment '{name}': {e}")

print("\nVerification of loaded checkpoints:")
if loaded_checkpoints:
    for exp_name, model_instance in loaded_checkpoints.items():
        print(f"  - '{exp_name}': {type(model_instance).__name__}")
else:
    print("No checkpoints were successfully loaded.")

Loading 5 checkpoints from /content/drive/MyDrive/butterflai_checkpoints...
Successfully loaded checkpoint for experiment 'E0'.
Successfully loaded checkpoint for experiment 'E1'.
Successfully loaded checkpoint for experiment 'E2'.
Successfully loaded checkpoint for experiment 'E3'.
Successfully loaded checkpoint for experiment 'E4'.

Verification of loaded checkpoints:
  - 'E0': ExtendedConditionalDiffusionLightning
  - 'E1': ExtendedConditionalDiffusionLightning
  - 'E2': ExtendedConditionalDiffusionLightning
  - 'E3': ExtendedConditionalDiffusionLightning
  - 'E4': ExtendedConditionalDiffusionLightning


In [17]:
import os
import shutil

# Source directory in Google Drive
source_dir = drive_checkpoint_dir

# Destination directory (local working directory where models are expected)
dest_dir = CKPT_DIR

# List of checkpoints to copy
checkpoints_to_copy = EXPERIMENT_NAMES_TO_LOAD

print(f"Copying checkpoints from '{source_dir}' to '{dest_dir}'...")

# Ensure the destination directory exists
os.makedirs(dest_dir, exist_ok=True)

for name in checkpoints_to_copy:
    source_file_name = f'ckpt_{name}.ckpt'
    source_path = os.path.join(source_dir, source_file_name)
    dest_path = os.path.join(dest_dir, source_file_name)

    if os.path.exists(source_path):
        shutil.copy(source_path, dest_path)
        print(f"  Copied {source_file_name}")
    else:
        print(f"  Warning: {source_file_name} not found in {source_dir}. Skipping.")

print("Checkpoint copying complete.")

Copying checkpoints from '/content/drive/MyDrive/butterflai_checkpoints' to '/content/butterflai/weeks/week_10'...
  Copied ckpt_E0.ckpt
  Copied ckpt_E1.ckpt
  Copied ckpt_E2.ckpt
  Copied ckpt_E3.ckpt
  Copied ckpt_E4.ckpt
Checkpoint copying complete.


<a id="eval-mode"></a>

---
# Evaluation mode — NLL ablations across all experiments (Tasks 67–70)

This is the evaluation half of the Week 10 extension. It scores every
checkpoint produced by train mode (`ckpt_E*.ckpt` in this directory)
against the same **hard-gated NLL** metric used in 10c, and adds a
critical new diagnostic: an **oracle MLP** that maps each experiment's
cond vector directly to per-bin Gaussian residual parameters. The
oracle's NLL is an *upper bound* on what any model can extract from a
given cond set:

- Diffusion ≪ oracle  →  architecture is the bottleneck (try FiLM, CFG, Fourier).
- Oracle ≈ classical  →  the cond set itself doesn't help; try a different
  cond group or stop running that variant.

This is what makes the ablation scientifically honest: without an
oracle, a flat NLL across experiments could mean *either* "more cond
doesn't help" *or* "the architecture can't extract the new cond's
information" — two completely different fixes.

*The code cells below only execute when `MODE == "eval"`.*


---
## Task 67 — Build per-window evaluation blocks

Re-window the raw CSV the same way 10c does (per-window, 6-monthly,
≥ 20 obs per window) and tag each window with its v2 parquet row's
*entire* cond superset — every group, normalized later per-experiment
using the corresponding checkpoint's `cond_<g>_means` / `cond_<g>_stds`
buffers. We work with **per-window blocks only** in 10e — that is the
granularity at which the diffusion model is native, and the granularity
where any improvement over Week 10 will be most visible.


In [18]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import timedelta

# Re-import ExtendedConditionalDiffusionLightning to ensure it's up to date after patching
from weeks.week_10.conditioned_infrastructure import ExtendedConditionalDiffusionLightning, ExtendedConditionalResidualDataset

if not EVAL_MODE:
    print("Skipping Task 67 Part 1 (per-window blocks) (MODE=train). Switch MODE to \"eval\" once you have ckpt_E*.ckpt files.")
else:
    # Task 67 — per-window blocks tagged with their v2 cond superset.

    # 1. Process raw_df: parse dates, compute abs_lat, derive hemisphere, drop missing CYCLE.
    raw_df = pd.read_csv(paths["raw_csv"])

    # Fix: Construct 'time_min' from existing 'year', 'month', 'day', 'hour', 'minute' columns
    # and handle potential errors or missing values by coercing to NaT.
    raw_df['time_min'] = pd.to_datetime(raw_df[['year', 'month', 'day', 'hour', 'minute']], errors='coerce')

    # Drop rows where 'CYCLE' is missing (as per original code)
    raw_df = raw_df.dropna(subset=['CYCLE'])
    raw_df['CYCLE'] = raw_df['CYCLE'].astype(int)

    # Also drop rows where 'time_min' parsing failed (resulting in NaT)
    raw_df = raw_df.dropna(subset=['time_min'])

    raw_df['abs_lat'] = raw_df['latitude'].abs()
    raw_df['hemisphere'] = np.where(raw_df['latitude'] >= 0, 'north', 'south')

    # Calculate year_decimal for raw_df to map to tau_center
    raw_df['year_decimal'] = raw_df['time_min'].dt.year + (raw_df['time_min'].dt.dayofyear - 1) / 365.25

    # Determine the min/max year_decimal from the raw data for tau_center un-normalization
    min_year_decimal_raw = raw_df['year_decimal'].min()
    max_year_decimal_raw = raw_df['year_decimal'].max()

    # Determine min/max tau_center from windows_aug for un-normalization
    min_tau_norm = windows_aug['tau_center'].min()
    max_tau_norm = windows_aug['tau_center'].max()

    # Define function to un-normalize tau_center to year_decimal
    def unnormalize_tau_center(tau_normalized):
        if max_tau_norm == min_tau_norm: # Handle case of single tau_center value
            return (max_year_decimal_raw + min_year_decimal_raw) / 2
        return ((tau_normalized - min_tau_norm) / (max_tau_norm - min_tau_norm)) * (max_year_decimal_raw - min_year_decimal_raw) + min_year_decimal_raw

    # 2. Define GROUP_COLS (see ExtendedConditionalDiffusionLightning.GROUP_COLS)
    # Copy from the class attribute to ensure it includes static definitions
    # Create a local GROUP_COLS dictionary first
    # This local GROUP_COLS is not the class attribute, but serves as a template
    GROUP_COLS = {
        "base":      ["amplitude", "tau_center", "mu_universal"],
        "cyclehemi": ["cycle_norm", "hemi_id"],
        "opp":       ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"],
        "traj":      [], # This will be filled dynamically
    }

    # "traj" cols are dynamically found from windows_aug
    traj_cols = [c for c in windows_aug.columns if c.startswith("area_lag")]
    GROUP_COLS["traj"] = traj_cols

    # --- Start of fix for AttributeError ---
    # Defensive check: Ensure GROUP_COLS is present on the class before attempting to modify it.
    # This handles cases where reloading might not have fully propagated the patched attribute.
    if not hasattr(ExtendedConditionalDiffusionLightning, 'GROUP_COLS'):
        ExtendedConditionalDiffusionLightning.GROUP_COLS = {
            "base":      ["amplitude", "tau_center", "mu_universal"],
            "cyclehemi": ["cycle_norm", "hemi_id"],
            "opp":       ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"],
            "traj":      [],
        }
        print("DEBUG: ExtendedConditionalDiffusionLightning.GROUP_COLS was missing and has been re-initialized.")

    if not hasattr(ExtendedConditionalResidualDataset, 'GROUP_COLS'):
        ExtendedConditionalResidualDataset.GROUP_COLS = {
            "base":      ["amplitude", "tau_center", "mu_universal"],
            "cyclehemi": ["cycle_norm", "hemi_id"],
            "opp":       ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"],
            "traj":      [],
        }
        print("DEBUG: ExtendedConditionalResidualDataset.GROUP_COLS was missing and has been re-initialized.")
    # --- End of fix for AttributeError ---

    # IMPORTANT: Update the class attributes for both classes
    # This ensures both classes use the consistent, dynamically discovered traj_cols
    ExtendedConditionalDiffusionLightning.GROUP_COLS["traj"] = traj_cols
    ExtendedConditionalResidualDataset.GROUP_COLS["traj"] = traj_cols # This is the crucial line to add

    # Ensure all GROUP_COLS columns are present in windows_aug
    for group, cols in GROUP_COLS.items():
        for col in cols:
            if col not in windows_aug.columns:
                raise ValueError(f"Column '{col}' for group '{group}' not found in windows_aug.")

    # 3. Build the hemicycles structure
    hemicycles = []
    # Iterate over unique (cycle, hemisphere, split) groups in windows_aug
    # Each unique (cycle, hemisphere, split) combo forms a hemicycle entry
    for (cycle, hemisphere, split), df_hemicycle_windows_aug in windows_aug.groupby(['cycle', 'hemisphere', 'split']):
        hemicycle_data = {
            "cycle": cycle,
            "hemisphere": hemisphere,
            "split": split,
            "blocks": [],
        }

        # Sort by tau_center to ensure correct ordering of blocks
        df_hemicycle_windows_aug = df_hemicycle_windows_aug.sort_values('tau_center')

        # Get amplitude and t0 for the hemicycle from its first window
        # Assuming amplitude is constant for a hemicycle
        hemicycle_data["amplitude"] = df_hemicycle_windows_aug.iloc[0]['amplitude']
        hemicycle_data["t0"] = df_hemicycle_windows_aug.iloc[0]['tau_center'] # normalized t0

        for _, window_aug_row in df_hemicycle_windows_aug.iterrows():
            block = {}
            block["tau"] = window_aug_row['tau_center'] # normalized tau_center from windows_aug

            # Un-normalize tau_center to get year_decimal_center
            year_decimal_center = unnormalize_tau_center(window_aug_row['tau_center'])
            block["center_decimal"] = year_decimal_center

            # Infer window boundaries (6-monthly window means +/- 0.25 years from center)
            window_start_year_decimal = year_decimal_center - 0.25
            window_end_year_decimal = year_decimal_center + 0.25

            # Filter raw_df to get lats within this window
            # Filter by cycle and hemisphere to be precise
            raw_lats_in_window = raw_df[
                (raw_df['CYCLE'] == cycle) &
                (raw_df['hemisphere'] == hemisphere) &
                (raw_df['year_decimal'] >= window_start_year_decimal) &
                (raw_df['year_decimal'] < window_end_year_decimal)
            ]['abs_lat'].values

            # Only add block if it meets the minimum observation count
            if len(raw_lats_in_window) >= 20: # 20 obs per window, as per 10c
                block["lats"] = raw_lats_in_window.astype(np.float32)

                # Extract groups_raw
                groups_raw = {}
                for group_name, cols in GROUP_COLS.items():
                    # Ensure the data type is float32 as expected by the model
                    groups_raw[group_name] = window_aug_row[cols].values.astype(np.float32)
                block["groups_raw"] = groups_raw
                hemicycle_data["blocks"].append(block)

        # Only add hemicycle if it has at least one valid block
        if hemicycle_data["blocks"]:
            hemicycles.append(hemicycle_data)


    assert len(hemicycles) > 0, "rebuild the per-window blocks before continuing"
    assert all("groups_raw" in blk for hc in hemicycles for blk in hc["blocks"]), (
        "every block needs a groups_raw dict")
    print(f"per-window blocks built: {sum(len(hc['blocks']) for hc in hemicycles)}")
    print(f"hemicycles included    : {len(hemicycles)}")


DEBUG: ExtendedConditionalDiffusionLightning.GROUP_COLS was missing and has been re-initialized.
per-window blocks built: 13
hemicycles included    : 13


---
## Task 67 (cont) — NLL primitives, same as 10c

These are byte-identical to the primitives in 10c. Re-stated here so
10e can be run standalone (without executing 10c first).

In [19]:
if not EVAL_MODE:
    print("Skipping Task 67 Parts 2-3 (hard NLL primitives) (MODE=train). Switch MODE to \"eval\" once you have ckpt_E*.ckpt files.")
else:
    # Task 67 — hard NLL primitives. Port from 10c (or import them if you've
    # factored them out). The two callables you need are:
    #
    #   hard_nll_classical(model, hcs) -> (nll, detail)
    #   hard_nll_combined (model, hcs, residuals_by_block, eps=1e-6)
    #                                                  -> (nll, detail)
    #
    # 10c defines both; they are byte-identical here. Once defined, the
    # print below should report a classical baseline near 10c's val number.

    # TODO: define hard_nll_classical and hard_nll_combined.

    def hard_nll_classical(model, hcs):
        """Evaluate hard NLL on an array of hemicycle structures. (10c version)"""
        num   = 0.0  # Numerator for NLL sum over blocks.
        den   = 0.0  # Denominator (number of raw observations).
        floor = 0    # Number of blocks where a full hard NLL couldn't be computed.

        # Each entry in `hcs` is one hemicycle (all blocks have same amplitude, etc).
        # Each `block` in `hcs[i]["blocks"]` is one window.
        for hemicycle in hcs:
            for block in hemicycle["blocks"]:
                # This is a dict with "lats" (array) and "groups_raw" (dict of arrays).
                # We only need the latitudes here.
                lats = block["lats"]

                # Get mean and std from the classical model.
                mean, std  = model.gaussian(hemicycle["amplitude"], block["tau"])

                # If the std is too small, the NLL goes to infinity. Count these
                # as "floored" blocks (and don't include them in the NLL calc).
                if std < 1e-3:
                    floor += 1
                    continue

                # Evaluate NLL of each latitude under the model's Gaussian.
                # `sp_norm.logpdf` is ln(likelihood). Summing gives ln(likelihood).
                # Dividing by num observations and negating gives NLL.
                ln_likelihood_per_obs = sp_norm.logpdf(lats, loc=mean, scale=std).sum()
                num += ln_likelihood_per_obs
                den += len(lats)
        return -num / den, {"coverage": 1.0 - floor / len(hcs), "floored_blocks": floor}

    from scipy.special import logsumexp

    def hard_nll_combined(model, hcs, residuals_by_block, eps=1e-6):
        """Evaluate hard NLL of combined classical + diffusion on an array of hemicycle structures.

        Args:
            model: The classical model (e.g., `classical` ButterflAIModel instance).
            hcs: List of hemicycle structures.
            residuals_by_block: List of K samples (physical latitude values) for each block.
                                Shape (K, 15) for each block, where 15 is len(BIN_CENTERS).
            eps: Small value to prevent log(0).

        Returns:
            (nll, detail_dict)
        """
        num   = 0.0  # Numerator for NLL sum over observations.
        den   = 0.0  # Denominator (number of raw observations).
        floor = 0    # Number of blocks where a full hard NLL couldn't be computed.

        residual_idx = 0
        for hemicycle in hcs:
            for block in hemicycle["blocks"]:
                lats = block["lats"]

                # Get mean and std from the classical model.
                classical_mean, classical_std  = model.gaussian(hemicycle["amplitude"], block["tau"])

                if classical_std < 1e-3:
                    floor += 1
                    residual_idx += 1 # Ensure we advance to the next residual set
                    continue

                # `diffusion_samples_physical` are K samples for the 15 bins, direct physical latitude values.
                # Shape (K, 15).
                diffusion_samples_physical = residuals_by_block[residual_idx]
                K_samples = diffusion_samples_physical.shape[0]

                log_likelihood_sum_for_block = 0.0
                for lat_obs in lats:
                    # Find the closest bin center for the current latitude observation.
                    # This determines which of the 15 dimensions of the diffusion samples to use.
                    bin_idx = np.argmin(np.abs(lat_obs - BIN_CENTERS))

                    # The `K` samples for this specific bin from the diffusion model.
                    # These are physical latitude predictions, not residuals.
                    bin_diffusion_predictions = diffusion_samples_physical[:, bin_idx]

                    # The combined PDF is a mixture: P(x) = (1/K) * sum_{k=1 to K} N(x | mu_k, sigma_classical)
                    # where mu_k are the `bin_diffusion_predictions`.
                    # We compute the log-likelihood of `lat_obs` under each of these K Gaussian components.
                    log_pdfs_for_samples = sp_norm.logpdf(lat_obs, loc=bin_diffusion_predictions, scale=classical_std)

                    # Combine log-likelihoods using logsumexp for numerical stability, then average.
                    # log( (1/K) * sum(exp(log_pdfs)) ) = logsumexp(log_pdfs) - log(K)
                    log_likelihood_lat_obs = logsumexp(log_pdfs_for_samples) - np.log(K_samples)
                    log_likelihood_sum_for_block += log_likelihood_lat_obs

                num += log_likelihood_sum_for_block
                den += len(lats)
                residual_idx += 1

        # Ensure we handled all residual blocks if there was a discrepancy (e.g., from floor)
        # This check might be too strict if floor skips blocks but residuals_by_block doesn't reflect that
        # assert residual_idx == len(residuals_by_block), "Mismatch in number of residual blocks processed"

        # Return NLL (negative average log-likelihood)
        return -num / den, {"coverage": 1.0 - floor / len(hcs), "floored_blocks": floor}

    # Calculate classical baseline NLL
    val_hcs = [hc for hc in hemicycles if hc["split"] == "val"]
    nll_cl_val, det_cl_val = hard_nll_classical(classical, val_hcs)
    print(f"classical hard NLL (val): {nll_cl_val:.4f}  "
          f"(coverage {det_cl_val["coverage"]:.3f})")

classical hard NLL (val): 3.1487  (coverage 1.000)


---
## Task 67 (cont) — Diagnostic oracle

For each experiment's cond set, fit a tiny MLP that maps the cond
vector directly to a 15-D Gaussian over the residual bins
(`mean`, `log_std`). The oracle's NLL is computed by sampling K
residuals from the per-block Gaussian and feeding them through
`hard_nll_combined` — the same harness used to score the diffusion.

What the oracle answers:

- **Diffusion ≪ oracle**  →  the diffusion isn't extracting the
  information that's already in the cond. Try a stronger architecture
  (FiLM, Fourier features, larger MLP).
- **Oracle ≈ classical**  →  the cond set itself doesn't carry enough
  information about the residual structure. Try a different cond
  group or stop adding to this one.

You implement this. The science (the tiny MLP, the Gaussian NLL
expression, the training loop, sampling from the predicted Gaussian)
is yours.

In [20]:
class OracleMLP(nn.Module):
    """Map a cond vector to per-bin Gaussian residual params (mean, log_std)."""
    def __init__(self, cond_dim, hidden_dim=64, num_bins=15):
        super().__init__()
        self.num_bins = num_bins
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * num_bins) # 15 means + 15 log_stds
        )

    def forward(self, cond):
        out = self.net(cond)
        mean = out[:, :self.num_bins]
        log_std = out[:, self.num_bins:]
        return mean, log_std

def gaussian_nll(r, mean, log_std, eps=1e-6):
    """Per-row, per-bin Gaussian NLL of the *standardized* residual ``r``
    under the predicted ``(mean, log_std)``. Return a scalar.

    r, mean, log_std are all (B, 15) tensors.
    Output is (B, 15) NLLs, then averaged across bins and batch.
    """
    # Clamp log_std to prevent too small stds leading to inf NLL
    log_std = torch.clamp(log_std, min=np.log(eps))
    std = torch.exp(log_std)
    # NLL = 0.5 * log(2*pi*sigma^2) + 0.5 * ((r - mu)/sigma)^2
    #     = log(sigma) + 0.5 * log(2*pi) + 0.5 * ((r - mu)/sigma)^2
    nll_per_bin = log_std + 0.5 * np.log(2 * np.pi) + 0.5 * (((r - mean) / std)**2)
    return nll_per_bin.mean()

def fit_oracle(cond_train, r_train, cond_val, r_val,
               max_epochs=500, lr=1e-2, hidden_dim=64, seed=0):
    """Fit OracleMLP on (cond_train, r_train); track val NLL each epoch
    and return the module with the best val state restored, along with
    the best val NLL. ``r_*`` are *standardized* residuals (15-D)."""
    torch.manual_seed(seed)
    cond_dim = cond_train.shape[-1]
    num_bins = r_train.shape[-1] # Should be 15

    oracle_mlp = OracleMLP(cond_dim, hidden_dim=hidden_dim, num_bins=num_bins).to(device) # Ensure model is on the correct device
    optimizer = torch.optim.Adam(oracle_mlp.parameters(), lr=lr)

    best_val_nll = float('inf')
    best_state_dict = None
    patience_counter = 0
    early_stopping_patience = 20 # Can be tuned

    # Convert numpy arrays to torch tensors
    cond_train_t = torch.tensor(cond_train, dtype=torch.float32).to(device)
    r_train_t = torch.tensor(r_train, dtype=torch.float32).to(device)
    cond_val_t = torch.tensor(cond_val, dtype=torch.float32).to(device)
    r_val_t = torch.tensor(r_val, dtype=torch.float32).to(device)

    print(f"Fitting oracle MLP (cond_dim={cond_dim}, hidden_dim={hidden_dim}, num_bins={num_bins})...")

    for epoch in range(max_epochs):
        oracle_mlp.train()
        optimizer.zero_grad()
        mean_pred, log_std_pred = oracle_mlp(cond_train_t)
        loss = gaussian_nll(r_train_t, mean_pred, log_std_pred)
        loss.backward()
        optimizer.step()

        oracle_mlp.eval()
        with torch.no_grad():
            val_mean_pred, val_log_std_pred = oracle_mlp(cond_val_t)
            val_nll = gaussian_nll(r_val_t, val_mean_pred, val_log_std_pred)

        if val_nll < best_val_nll:
            best_val_nll = val_nll
            best_state_dict = oracle_mlp.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= early_stopping_patience:
            # print(f"  Early stopping at epoch {epoch} (best val NLL: {best_val_nll:.4f})")
            break
    # print(f"  Finished training. Final best val NLL: {best_val_nll:.4f}")

    if best_state_dict is not None:
        oracle_mlp.load_state_dict(best_state_dict)

    # Ensure the returned best_val_nll is a standard Python float
    if isinstance(best_val_nll, torch.Tensor):
        best_val_nll = best_val_nll.item()

    return oracle_mlp, best_val_nll

---
## Task 68 — Score every checkpoint

For each discovered `ckpt_E*.ckpt`:

1. Build per-experiment cond tensors for every val block by
   concatenating the right groups in `consumed_keys` order, normalized
   with the **checkpoint's own** per-group buffers (so val data uses
   train-set normalization recovered from the saved model).
2. Run K = 100 conditional samples per block using
   `sample_conditional_extended`. For E6 (CFG), repeat the sampling at
   every guidance weight in `CFG_GUIDANCE_W` and keep them as separate
   rows.
3. Fit the oracle MLP on the same (cond, standardized residual) data
   and record its val NLL as the upper bound for this cond set.
4. Plug each of the K samples into `hard_nll_combined`; report mean
   and σ over K.

In [45]:
K = 300   # K samples per val block; bump to 500 for tighter K-σ bars.

# Guidance values to sweep when scoring a CFG-trained checkpoint.
CFG_GUIDANCE_W = [1.0, 1.5, 2.0, 3.0]


def score_checkpoint(name, cfg):
    """Score one experiment.

    The recipe:
      1. Load the trained model + datasets via
         `load_trained_experiment(name, cfg, windows_aug, _WEEK10_DIR,
                                  alpha_np, sigma_np)`.
      2. Build per-val-block cond tensors via
         `block_cond_concat(val_hcs, lit, cfg, train_ds)`,
         repeat each row K times, and sample residuals with
         `sample_conditional_extended(lit, cond_K, guidance_w=w,
                                       device=device)`.
         If `cfg["cond_dropout_p"] > 0`, sweep every `w` in
         `CFG_GUIDANCE_W` and emit one row per `w`; otherwise sample
         once at `w = 0.0`.
      3. Reshape samples to (N, K, 15) physical units; push them through
         `k_run_combined(hard_nll_combined, classical, val_hcs, keys,
                          samples_NK15)` to get K NLLs and floor
         fractions.
      4. Fit the oracle on the same cond set: build (cond, standardized residual)
         pairs for train and val blocks, standardize residuals with the
         train-set bin stats, and call `fit_oracle`. Convert the
         oracle's per-block Gaussian into K physical-unit samples and
         push them through `k_run_combined` to get the oracle's
         hard-NLL upper bound.

    Returns a list of dicts with keys
    {experiment, guidance_w, nll_mean, nll_std, floor,
     oracle_nll_mean, oracle_nll_std, oracle_gauss, coverage}.
    """

    all_results = []

    # 1. Load the trained model + datasets
    lit_model, diffusion_model, val_ds, train_ds = load_trained_experiment(
        name, cfg, windows_aug, CKPT_DIR, alpha_np, sigma_np)
    lit_model.eval().to(device)

    print(f"DEBUG: Scoring experiment: {name}")
    print(f"DEBUG: Loaded model hparams for {name}: {lit_model.hparams}")

    # Inspect the model's expected input dimension
    model_expected_in_features = -1
    cond_dim_expected_by_model = -1

    # Try to find the first Linear layer that processes the concatenated input
    # This assumes ExtendedConditionalDiffusionModel has an input_linear for concat or a primary linear layer.
    model_input_layer = None
    if hasattr(lit_model.model, 'input_linear'): # For concat architecture
        model_input_layer = lit_model.model.input_linear
    else: # Attempt to find first linear layer, common in sequential models
        for module in lit_model.model.modules():
            if isinstance(module, nn.Linear):
                model_input_layer = module
                break

    if model_input_layer is not None:
        model_expected_in_features = model_input_layer.in_features
        try:
            data_dim = lit_model.model.data_dim # Should be 15
            # time_mlp is a Sequential, its last module's out_features is the dim
            time_emb_dim = lit_model.model.time_mlp[-1].out_features
            cond_dim_expected_by_model = model_expected_in_features - data_dim - time_emb_dim
            print(f"DEBUG: Model expected total input features (from first linear layer): {model_expected_in_features}")
            print(f"DEBUG: Model expected data_dim: {data_dim}, time_emb_dim: {time_emb_dim}")
            print(f"DEBUG: Model expected cond_dim: {cond_dim_expected_by_model}")
        except AttributeError:
            print("DEBUG: Could not fully determine expected cond_dim from model structure.")
    else:
        print("DEBUG: Could not find a suitable input linear layer in the model to determine expected input features.")

    # Global val_hcs is already filtered for split="val"
    # Ensure it's not empty
    if not val_hcs:
        print(f"No validation hemicycles found for experiment {name}. Skipping scoring.")
        return []

    # 2. Build per-val-block cond tensors and sample residuals.
    # `block_cond_concat` returns a list of tensors, where each tensor is (num_items_in_block, cond_dim).
    # For our current setup, num_items_in_block is 1.
    val_cond_tensors_per_block = block_cond_concat(val_hcs, lit_model, cfg)

    # Calculate num_val_blocks
    num_val_blocks = sum(len(hc['blocks']) for hc in val_hcs)

    if not val_cond_tensors_per_block or num_val_blocks == 0:
        print(f"No conditional tensors or validation blocks found for experiment {name}. Skipping sampling.")
        return []

    # Concatenate the list of (1, cond_dim) tensors into (num_val_blocks, cond_dim)
    all_val_cond_for_sampling = torch.cat(val_cond_tensors_per_block, dim=0).to(device)
    actual_cond_dim_from_block_cond_concat = all_val_cond_for_sampling.shape[-1]
    print(f"DEBUG: Actual cond_dim produced by block_cond_concat: {actual_cond_dim_from_block_cond_concat}")

    if cond_dim_expected_by_model != -1 and actual_cond_dim_from_block_cond_concat != cond_dim_expected_by_model:
        print(f"ERROR: Mismatch in cond_dim for {name}! Expected: {cond_dim_expected_by_model}, Actual: {actual_cond_dim_from_block_cond_concat}")
        # If we uncomment the next line, execution will stop here, allowing for easier debugging.
        # raise ValueError(f"Cond_dim mismatch for {name}: Expected {cond_dim_expected_by_model}, Got {actual_cond_dim_from_block_cond_concat}")

    # Prepare for conditional sampling
    guidance_weights_to_test = [0.0]
    if cfg.get("cond_dropout_p", 0.0) > 0.0:
        guidance_weights_to_test = CFG_GUIDANCE_W

    for guidance_w in guidance_weights_to_test:
        # Repeat each block's condition K times for sampling
        # Shape: (num_val_blocks * K, cond_dim)
        cond_K = repeat(all_val_cond_for_sampling, 'n d -> (n k) d', k=K)

        # Sample residuals (standardized)
        samples_std = sample_conditional_extended(
            lit_model, cond_K, guidance_w=guidance_w, device=device
        ).detach().cpu().numpy() # Shape: (num_val_blocks * K, 15)

        # Convert standardized samples to physical units
        samples_phys = samples_std * val_ds.bin_stds.numpy() + val_ds.bin_means.numpy() # Shape: (num_val_blocks * K, 15)

        # Reshape samples into a list of (K, 15) arrays for k_run_combined
        diffusion_samples_for_blocks = [
            samples_phys[i * K : (i + 1) * K, :] for i in range(num_val_blocks)
        ]

        # 3. Calculate Diffusion NLL
        diff_nll_mean, diff_nll_std, diff_floor, diff_coverage = k_run_combined(
            hard_nll_combined, classical, val_hcs,
            diffusion_samples_for_blocks
        )

        # 4. Fit the oracle and calculate its NLL.
        # Prepare (cond, standardized residual) pairs for oracle training
        # Train data for oracle
        train_cond_list = []
        train_r_list = []
        for i in range(len(train_ds)):
            item = train_ds[i]
            cond_vec = torch.cat([item[k] for k in cfg["consumed_keys"]]).unsqueeze(0)
            train_cond_list.append(cond_vec)
            train_r_list.append(item["r_clean"].unsqueeze(0))

        if not train_cond_list: # Handle empty train_ds case
            cond_train_oracle = np.array([])
            r_train_oracle = np.array([])
        else:
            cond_train_oracle = torch.cat(train_cond_list, dim=0).cpu().numpy()
            r_train_oracle = torch.cat(train_r_list, dim=0).cpu().numpy()

        # Val data for oracle
        val_cond_list = []
        val_r_list = []
        for i in range(len(val_ds)):
            item = val_ds[i]
            cond_vec = torch.cat([item[k] for k in cfg["consumed_keys"]]).unsqueeze(0)
            val_cond_list.append(cond_vec)
            val_r_list.append(item["r_clean"].unsqueeze(0))

        if not val_cond_list: # Handle empty val_ds case
            cond_val_oracle = np.array([])
            r_val_oracle = np.array([])
        else:
            cond_val_oracle = torch.cat(val_cond_list, dim=0).cpu().numpy()
            r_val_oracle = torch.cat(val_r_list, dim=0).cpu().numpy()

        # Ensure that there is data to train/evaluate the oracle.
        if cond_train_oracle.size == 0 or cond_val_oracle.size == 0 or cond_train_oracle.ndim == 1 or cond_val_oracle.ndim == 1:
            print(f"Not enough data to train/evaluate oracle for experiment {name}. Skipping oracle NLL.")
            oracle_nll_mean, oracle_nll_std, oracle_floor, oracle_coverage, best_val_nll_oracle = (np.nan, np.nan, np.nan, np.nan, np.nan)
        else:
            # Fit the oracle MLP
            oracle_mlp, best_val_nll_oracle = fit_oracle(
                cond_train_oracle, r_train_oracle,
                cond_val_oracle, r_val_oracle,
                hidden_dim=cfg["hidden_dim"]
            )
            oracle_mlp.eval()

            # Generate K physical samples from the oracle's Gaussian for each validation block
            oracle_samples_for_blocks = []
            with torch.no_grad():
                for cond_vec in torch.tensor(cond_val_oracle, dtype=torch.float32).to(device):
                    mean, log_std = oracle_mlp(cond_vec.unsqueeze(0))
                    std = torch.exp(log_std)

                    epsilon = torch.randn(K, len(BIN_CENTERS), device=device)
                    oracle_samples_std_k15 = mean + std * epsilon
                    oracle_samples_phys_k15 = oracle_samples_std_k15.cpu().numpy() * val_ds.bin_stds.numpy() + val_ds.bin_means.numpy()
                    oracle_samples_for_blocks.append(oracle_samples_phys_k15)

            # Calculate Oracle NLL using k_run_combined
            oracle_nll_mean, oracle_nll_std, oracle_floor, oracle_coverage = k_run_combined(
                hard_nll_combined, classical, val_hcs,
                oracle_samples_for_blocks
            )

        all_results.append({
            "experiment": name,
            "guidance_w": guidance_w,
            "nll_mean": diff_nll_mean,
            "nll_std": diff_nll_std,
            "floor": diff_floor, # Floor from diffusion calculation
            "oracle_nll_mean": oracle_nll_mean,
            "oracle_nll_std": oracle_nll_std,
            "oracle_gauss": best_val_nll_oracle,
            "coverage": diff_coverage # Coverage from diffusion calculation
        })
    return all_results

In [48]:
if not EVAL_MODE:
    print("Skipping Task 68 scoring loop (MODE=train). Switch MODE to \"eval\" once you have ckpt_E*.ckpt files.")
else:
    # Discover trained checkpoints and score them.
    _ckpts = discover_experiment_checkpoints(_WEEK10_DIR)
    for name in _ckpts:
        assert name in EXPERIMENTS, (
            f"ckpt_{name}.ckpt has no entry in EXPERIMENTS — add a spec to Task 65 "
            f"(both halves of this notebook share the same dict)."
        )
    print(f"discovered checkpoints: {list(_ckpts)}")

    all_rows = []
    for name in _ckpts:
        print(f"scoring {name} ...")
        all_rows.extend(score_checkpoint(name, EXPERIMENTS[name]))

    scoreboard = pd.DataFrame(all_rows)
    scoreboard["classical"] = nll_cl_val
    print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

discovered checkpoints: ['E0', 'E1', 'E2', 'E3', 'E4']
scoring E0 ...


RuntimeError: Error(s) in loading state_dict for ExtendedConditionalDiffusionLightning:
	Missing key(s) in state_dict: "model.fourier.freqs". 
	size mismatch for model.net.0.weight: copying a param with shape torch.Size([128, 147]) from checkpoint, the shape in current model is torch.Size([128, 179]).

---
## Task 69 — Headline plot

Bar chart, val split: classical baseline plus every experiment's NLL,
with K-σ error bars. Oracle NLL per experiment overlaid as a
horizontal dashed marker to make the "what's achievable from this cond
set" boundary visible.

For any CFG variant (`cond_dropout_p > 0`), the bar shown is the
best-NLL guidance setting; a secondary panel sweeps the guidance
weight `w` so you can see the guidance vs NLL trade.

In [113]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import logsumexp

if not EVAL_MODE:
    print("Skipping Task 69 headline plot (MODE=train). Switch MODE to \"eval\" once you have ckpt_E*.ckpt files.")
else:
    # Task 69 — headline plot + CFG sweep + per-hemicycle breakdown.
    #
    # Required panels:
    #   1. Bar chart of val NLL: classical bar on the left, one bar per
    #      experiment (best guidance w if it's a CFG variant), with K-σ
    #      error bars. Overlay each experiment's oracle NLL as a dashed
    #      horizontal marker so the "what's achievable from this cond set"
    #      boundary is visible.
    #   2. (If any CFG checkpoint exists) a sweep of guidance weight vs NLL
    #      on the CFG variant.
    #   3. Per-hemicycle breakdown for the best variant — same axes as the
    #      Week 10 chart so any improvement is visually unambiguous.
    #
    # Useful values you already have:
    #   - `scoreboard` (DataFrame from Task 68)
    #   - `nll_cl_val` (classical baseline)
    #   - `val_hcs`, `EXPERIMENTS`, `K`, `_WEEK10_DIR`
    #
    # For panel 3, reuse `load_trained_experiment(...)`,
    # `block_cond_concat(...)`, `sample_conditional_extended(...)`, and
    # `k_run_combined(...)`.

    # --- Panel 1: Headline NLL Plot (Bar Chart) ---

    # Prepare data for plotting
    plot_data = scoreboard.copy()

    # Find the best NLL for each experiment (considering guidance if multiple values exist)
    # For simplicity, if there are multiple guidance_w, we'll pick the one with the min nll_mean.
    # If there's no guidance, it will just pick the single entry.
    # Ensure scoreboard is not empty before grouping
    if not plot_data.empty:
        idx = plot_data.groupby('experiment')['nll_mean'].idxmin()
        best_nll_per_experiment = plot_data.loc[idx].reset_index(drop=True)
    else:
        best_nll_per_experiment = pd.DataFrame() # Create empty if scoreboard was empty

    # Add the classical baseline as a row for plotting
    classical_row = pd.DataFrame(
        [
            {
                'experiment': 'Classical',
                'guidance_w': 0.0,
                'nll_mean': nll_cl_val,
                'nll_std': 0.0, # Classical NLL is a single value, so std is 0.0 for this context
                'floor': 0.0,
                'oracle_nll_mean': np.nan, # No oracle for classical
                'oracle_nll_std': np.nan,
                'oracle_gauss': np.nan,
                'coverage': det_cl_val['coverage']
            }
        ]
    )

    plot_df = pd.concat([classical_row, best_nll_per_experiment]).reset_index(drop=True)

    fig_headline, ax_headline = plt.subplots(figsize=(10, 6))

    sns.barplot(
        x='experiment',
        y='nll_mean',
        data=plot_df,
        ax=ax_headline,
        palette='viridis',
        hue='experiment',
        legend=False,
        errorbar=None      # Explicitly disable seaborn's errorbar calculation
    )

    # Manually add error bars
    # Get the x-coords for the center of each bar
    x_coords = [p.get_x() + p.get_width() / 2 for p in ax_headline.patches]
    y_values = plot_df['nll_mean'].to_numpy()
    yerr_values = plot_df['nll_std'].to_numpy()

    ax_headline.errorbar(
        x=x_coords,
        y=y_values,
        yerr=yerr_values,
        fmt='none',  # Do not connect the error bars with a line
        color='black',
        capsize=5
    )

    # Overlay oracle NLLs as horizontal dashed lines
    for i, row in plot_df.iterrows():
        if pd.notna(row['oracle_nll_mean']) and row['experiment'] != 'Classical':
            # Use a more consistent way to get color if palette is used
            color_map = plt.cm.get_cmap('viridis', len(plot_df))
            bar_color = color_map(i)
            ax_headline.axhline(
                row['oracle_nll_mean'],
                color=bar_color,
                linestyle='--',
                linewidth=1.5,
                label=f'Oracle {row["experiment"]}' if i == 0 else "_nolegend_" # Only label once
            )

    ax_headline.set_title('Headline NLL Comparison (Validation Split)')
    ax_headline.set_xlabel('Experiment')
    ax_headline.set_ylabel('Mean NLL')
    ax_headline.grid(axis='y', linestyle='--', alpha=0.7)
    # Recreate legend to include custom oracle lines
    handles, labels = ax_headline.get_legend_handles_labels()
    # Filter out duplicate labels from hue and add oracle lines if any
    unique_labels = list(dict.fromkeys(labels))
    unique_handles = []
    for lab in unique_labels:
        for h, l in zip(handles, labels):
            if l == lab:
                unique_handles.append(h)
                break

    # Add specific oracle labels from the loop if they exist (only the first one for legend)
    oracle_labels_added = False
    for i, row in plot_df.iterrows():
        if pd.notna(row['oracle_nll_mean']) and row['experiment'] != 'Classical':
            if not oracle_labels_added:
                unique_handles.append(plt.Line2D([0], [0], color='black', linestyle='--', linewidth=1.5))
                unique_labels.append('Oracle NLL')
                oracle_labels_added = True
            break # Only add one oracle legend entry

    ax_headline.legend(unique_handles, unique_labels, title='Legend', loc='upper right')
    plt.tight_layout()
    plt.show()

NameError: name 'scoreboard' is not defined

In [109]:
# --- Panel 2: CFG Sweep (if any CFG checkpoint exists) ---

# Check if any experiment used cond_dropout_p > 0
cfg_experiments = [name for name, cfg_spec in EXPERIMENTS.items() if cfg_spec['cond_dropout_p'] > 0]

if cfg_experiments:
    print(f"Found CFG experiments: {cfg_experiments}. Generating CFG sweep plot.")
    # Filter scoreboard for CFG experiments
    # Ensure scoreboard is not empty before filtering
    if not scoreboard.empty:
        cfg_scoreboard = scoreboard[scoreboard['experiment'].isin(cfg_experiments)].copy()
    else:
        cfg_scoreboard = pd.DataFrame() # Create empty if scoreboard was empty

    if not cfg_scoreboard.empty:
        fig_cfg, ax_cfg = plt.subplots(figsize=(10, 6))

        for exp_name in cfg_scoreboard['experiment'].unique():
            exp_data = cfg_scoreboard[cfg_scoreboard['experiment'] == exp_name]
            ax_cfg.plot(exp_data['guidance_w'], exp_data['nll_mean'], marker='o', label=f'Experiment {exp_name}')
            ax_cfg.fill_between(
                exp_data['guidance_w'],
                exp_data['nll_mean'] - exp_data['nll_std'],
                exp_data['nll_mean'] + exp_data['nll_std'],
                alpha=0.2
            )

        ax_cfg.set_title('CFG Guidance Weight Sweep vs. NLL')
        ax_cfg.set_xlabel('Guidance Weight (w)')
        ax_cfg.set_ylabel('Mean NLL')
        ax_cfg.grid(True, linestyle='--', alpha=0.7)
        ax_cfg.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No data found in scoreboard for CFG experiments to plot.")
elif not cfg_experiments: # Corrected condition to check if no CFG experiments were defined
    print("No CFG experiments found. Skipping CFG sweep plot.")
else:
    print("No CFG experiments found. Skipping CFG sweep plot.")

No CFG experiments found. Skipping CFG sweep plot.


In [ ]:
from scipy.special import logsumexp

# Helper to calculate classical NLL for a single block
def hard_nll_classical_single_block(model, hemicycle, block_data, eps=1e-3):
    lats = block_data["lats"]
    # model.gaussian expects hemicycle amplitude and block tau
    mean, std = model.gaussian(hemicycle["amplitude"], block_data["tau"])
    if std < eps:
        return np.nan # Indicate floored
    ln_likelihood_per_obs = sp_norm.logpdf(lats, loc=mean, scale=std).sum()
    return -ln_likelihood_per_obs / len(lats) if len(lats) > 0 else np.nan

# Helper to calculate combined NLL for a single block, given the diffusion samples for that block
def hard_nll_combined_single_block(model, hemicycle, block_data, samples_K15_for_block, eps=1e-6):
    lats = block_data["lats"]
    classical_mean, classical_std = model.gaussian(hemicycle["amplitude"], block_data["tau"])

    if classical_std < eps:
        return np.nan

    diffusion_samples_physical = samples_K15_for_block # (K, 15)
    K_samples = diffusion_samples_physical.shape[0]

    log_likelihood_sum_for_block = 0.0
    for lat_obs in lats:
        bin_idx = np.argmin(np.abs(lat_obs - BIN_CENTERS))
        bin_diffusion_predictions = diffusion_samples_physical[:, bin_idx]
        log_pdfs_for_samples = sp_norm.logpdf(lat_obs, loc=bin_diffusion_predictions, scale=classical_std)
        log_likelihood_lat_obs = logsumexp(log_pdfs_for_samples) - np.log(K_samples)
        log_likelihood_sum_for_block += log_likelihood_lat_obs

    return -log_likelihood_sum_for_block / len(lats) if len(lats) > 0 else np.nan


# Identify the best performing experiment (lowest NLL mean, excluding classical and oracle)
# Ensure scoreboard is not empty before proceeding
if not scoreboard.empty:
    # Filter out 'Classical' row from scoreboard before finding best experiment
    filtered_scoreboard = scoreboard[scoreboard['experiment'] != 'Classical']
    if not filtered_scoreboard.empty:
        best_exp_row = filtered_scoreboard.loc[filtered_scoreboard['nll_mean'].idxmin()]
        best_exp_name = best_exp_row['experiment']
        best_exp_cfg = EXPERIMENTS[best_exp_name]

        print(f"Generating per-hemicycle breakdown for best experiment: {best_exp_name}")

        # Reload the best model and relevant datasets
        lit_best, _, val_ds_best, train_ds_best = load_trained_experiment(
            best_exp_name, best_exp_cfg, windows_aug, CKPT_DIR, alpha_np, sigma_np
        )
        lit_best.eval().to(device)

        # Get conditional inputs for all validation blocks
        raw_val_cond_blocks_best = block_cond_concat(val_hcs, lit_best, best_exp_cfg, train_ds_best)

        # Prepare flattened list of conditions for sampling
        flattened_val_cond_tensors_best = []
        for block_tensor in raw_val_cond_blocks_best:
            for item_cond_vector in block_tensor:
                flattened_val_cond_tensors_best.append(item_cond_vector.unsqueeze(0))

        all_samples_best = []
        for item_cond_vector in flattened_val_cond_tensors_best:
            cond_tensor_repeated_K = repeat(item_cond_vector.to(device), 'n d -> (n k) d', k=K)
            # Use best_exp_row['guidance_w'] for sampling
            guidance_w_for_sampling = best_exp_row['guidance_w'] if 'guidance_w' in best_exp_row else 0.0
            samples_std = sample_conditional_extended(
                lit_best, cond_tensor_repeated_K, guidance_w=guidance_w_for_sampling, device=device
            ).detach().cpu().numpy()
            samples_phys = samples_std * val_ds_best.bin_stds.numpy() + val_ds_best.bin_means.numpy()
            all_samples_best.append(samples_phys)

        samples_NK15_best = np.stack(all_samples_best)

        # Calculate per-block NLLs for plotting using the new helper functions
        per_block_nlls_diff = []
        block_idx_counter = 0
        block_labels = []
        block_nlls_classical = []

        for hc in val_hcs:
            for block_data in hc['blocks']:
                # Diffusion NLL for this block
                samples_K15_for_this_block = samples_NK15_best[block_idx_counter]
                nll_diff_block = hard_nll_combined_single_block(classical, hc, block_data, samples_K15_for_this_block)
                per_block_nlls_diff.append(nll_diff_block)

                # Classical NLL for this block
                nll_cl_block = hard_nll_classical_single_block(classical, hc, block_data)
                block_nlls_classical.append(nll_cl_block)

                block_labels.append(f"C{hc['cycle']}_{hc['hemisphere'][0].upper()}_{block_data['tau']:.1f}")
                block_idx_counter += 1

        # Create a DataFrame for easy plotting
        breakdown_df = pd.DataFrame({
            'Block': block_labels,
            'Classical NLL': block_nlls_classical,
            f'{best_exp_name} NLL': per_block_nlls_diff,
        })
        breakdown_df = breakdown_df.set_index('Block')

        fig_breakdown, ax_breakdown = plt.subplots(figsize=(14, 7))
        breakdown_df.plot(kind='bar', ax=ax_breakdown, alpha=0.8)
        ax_breakdown.set_title(f'Per-Hemicycle NLL Breakdown (Validation Split) - {best_exp_name} vs Classical')
        ax_breakdown.set_xlabel('Hemicycle Block')
        ax_breakdown.set_ylabel('Mean NLL')
        ax_breakdown.tick_params(axis='x', rotation=45)
        ax_breakdown.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print("No valid experiments in scoreboard to determine the best experiment for breakdown plot.")
else:
    print("Scoreboard is empty. Skipping Panel 3 plot.")

---
## Task 70 — Going further

Once you've worked through the Level 1–5 escalation menu in 10d and
want to push further, the tiered menu below ranks the next experiments
by expected payoff per unit effort. Discipline still applies: one knob
at a time, log to wandb, add to `EXPERIMENTS` in both 10d and 10e,
then re-run.

**Level 1 — easy wins**
- **Wider/deeper MLP.** Bump `hidden_dim` from 128 to 256, `n_layers`
  from 3 to 5 in the winning experiment's config. If NLL drops, the
  network was capacity-bound — interesting on its own.
- **Longer K at evaluation.** Bump K from 100 to 500 for the winning
  variant — tightens the K-σ error bar and lets you trust smaller
  margins.
- **Sampler comparison.** Re-score the winner with DDPM (stochastic)
  sampling instead of the deterministic DDIM in
  `sample_conditional_extended`. Deterministic samplers can under-
  disperse, inflating NLL.

**Level 2 — extra cond information**
- **Larger trajectory K.** Bump `K_LAGS` from 4 to 8 in 10d. If the
  trajectory variant's oracle improves but its diffusion doesn't, the
  architecture is underusing the longer history.
- **Lagged opposite-hemisphere.** Pair the trajectory cond with the
  opposite hemisphere — `opp_area_smoothed_lag1..lag4`.

**Level 3 — architectural changes**
- **Cross-attention conditioning.** Replace the FiLM mechanism with
  cross-attention over a small set of learned cond tokens — overkill
  for the cond dim here, but worth knowing if the FiLM gain saturates.
- **Per-bin-aware loss.** Weight the ε-prediction loss by the inverse
  per-bin std so well-resolved bins don't dominate gradients.

**The test set is the PI's.** Every iteration above is val-only. The
final test-set reveal happens once, after the program is closed.

---
## Handoff back to the PI

The headline numbers in `scoreboard` answer two questions:

1. **Does any variant beat the classical baseline on val?** If yes, the
   diffusion approach has earned its place in the final pipeline.
2. **Where is the bottleneck — information or architecture?** Compare
   each row's `nll_mean` to its `oracle_nll_mean`. A large gap means
   the cond set has more information than the diffusion is extracting
   (architecture-bound). A small gap with the oracle near classical
   means the cond set isn't carrying enough information — that line of
   experiments is exhausted; try a different cond group.

The PI will run the test set evaluation on whatever variant the val
results recommend.
